[目录](./table_of_contents.ipynb)

# 集合卡尔曼滤波器

In [ ]:
%matplotlib inline

In [ ]:
#格式化本书
import book_format
book_format.set_style()

> 我对集成滤波器不是很熟悉。我为这本书实现了一个滤波器，并使其工作，但我在现实生活中从未使用过。不同的来源使用稍微不同的这些方程式。如果我实现源代码中给出的方程式，滤波器就无法正常工作。可能是我做错了什么。然而，在网络上的各个地方，我看到了一些人发表评论，说明他们做了我在滤波器中做的事情，以使其正常工作。简而言之，我不太理解这个主题，但选择呈现我的知识缺失，而不是掩盖这个主题。我希望将来能够掌握这个主题，并撰写一个更具权威性的章节。在本章的结尾，我记录了我目前的困惑和问题。无论如何，如果我被源代码搞糊涂了，也许你也会，因此记录我的困惑可以帮助你避免相同的错误。

集合卡尔曼滤波器（EnKF）与上一章的无迹卡尔曼滤波器（UKF）非常相似。如果您还记得，UKF使用一组确定性选择的带权sigma点通过非线性状态和测量函数。在sigma点通过函数后，我们找到点的均值和协方差，并将其用作滤波器的新均值和协方差。它只是真实值的近似，因此次优，但在实践中，该滤波器非常准确。它的优点在于通常比EKF产生更准确的估计，并且不需要您分析地导出状态和测量方程的线性化。

集合卡尔曼滤波器的工作方式类似，只是它使用*蒙特卡罗*方法选择大量的sigma点。它起源于地球物理科学，作为对建模海洋和大气等物体所需的非常大的状态和系统的答案。有一篇有趣的文章介绍了它在天气建模中的发展 *[1]* 。该滤波器首先通过随机生成围绕滤波器初始状态的大量点来启动。该分布与滤波器的协方差$\mathbf{P}$成比例。换句话说，68%的点将在均值的一个标准偏差内，95%的点将在两个标准偏差内，依此类推。让我们在二维中观察一下这个问题。我们将使用`numpy.random.multivariate_normal()`函数从均值为(5,3)，协方差矩阵为

$$\begin{bmatrix}
32 & 15 \\ 15 & 40
\end{bmatrix}$$

的多元正态分布中随机创建点。

我画出了两个标准差的协方差椭圆来说明点的分布情况。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy.random import multivariate_normal
from filterpy.stats import (covariance_ellipse, 
                            plot_covariance_ellipse)

mean = (5, 3)
P = np.array([[32, 15],
              [15., 40.]])

x,y = multivariate_normal(mean=mean, cov=P, size=2500).T
plt.scatter(x, y, alpha=0.3, marker='.')
plt.axis('equal')

plot_covariance_ellipse(mean=mean, cov=P,
                        variance=2.**2,
                        facecolor='none')

## 算法

正如我之前所述，当滤波器初始化时，会从初始状态（$\mathbf{x}$）和协方差（$\mathbf{P}$）中绘制大量的 sigma 点。从那里开始，算法的步骤与 UKF 非常相似。在预测步骤中，sigma 点通过状态转移函数传递，然后添加一些噪声以考虑过程噪声。在更新步骤中，sigma 点通过测量函数转化为测量空间，它们会被微小地扰动以考虑测量噪声。卡尔曼增益从中计算出来。

我们已经提到了 UKF 和 EnKF 之间的主要区别——UKF 是确定性地选择 sigma 点。这里还有另一个区别，来自上述算法的暗示。在 UKF 中，我们在每个预测步骤中生成新的 sigma 点，并通过使用*不散变换*将这些点通过非线性函数重新组合成均值和协方差。EnKF 一直传播最初创建的 sigma 点；我们只需要计算一个均值和协方差作为滤波器的输出！

让我们看看滤波器的方程式。和往常一样，我会省略掉典型的下标和上标；我在表达一个算法，而不是数学函数。这里 $N$ 是 sigma 点的数量，$\chi$ 是 sigma 点的集合。

### 初始化步骤

$$\boldsymbol\chi \sim \mathcal{N}(\mathbf{x}_0, \mathbf{P}_0)$$

这意味着从滤波器的初始均值和协方差中选择 sigma 点。在代码中，这可能看起来像是：

In [ ]:
N = 1000
sigmas = multivariate_normal(mean=x, cov=P, size=N)

### 预测步骤

$$
\begin{aligned}
\boldsymbol\chi &= f(\boldsymbol\chi, \mathbf{u}) + v_Q \\
\mathbf{x} &= \frac{1}{N} \sum_1^N \boldsymbol\chi
\end{aligned}
$$

这段话简短明了，但可能不太清晰。第一行将所有 sigma 点通过用户提供的状态转移函数传递，然后添加一些根据 $\mathbf{Q}$ 矩阵分布的噪声。在 Python 中，我们可以这样写：

In [ ]:
for i, s in enumerate(sigmas):
    sigmas[i] = fx(x=s, dt=0.1, u=0.)

sigmas += multivariate_normal(x, Q, N)

第二行从 sigma 点计算平均值。在 Python 中，我们将利用 `numpy.mean` 来非常简洁快速地完成这项工作。

In [ ]:
x = np.mean(sigmas, axis=0)

现在我们可以选择性地计算均值的协方差。算法不需要计算这个值，但是它经常用于分析。该方程式如下：

$$\mathbf{P} = \frac{1}{N-1}\sum_1^N[\boldsymbol\chi-\mathbf{x}^-][\boldsymbol\chi-\mathbf{x}^-]^\mathsf{T}$$

其中，$\boldsymbol\chi-\mathbf{x}^-$ 是一维向量，因此我们将使用 `numpy.outer` 来计算 $[\boldsymbol\chi-\mathbf{x}^-][\boldsymbol\chi-\mathbf{x}^-]^\mathsf{T}$ 项。在Python中，我们可以编写：

In [ ]:
    P = 0
    for s in sigmas:
        P += outer(s-x, s-x)
    P = P / (N-1)

### 更新步骤

在更新步骤中，我们通过测量函数传递 sigma 点，计算 sigma 点的均值和协方差，从协方差计算卡尔曼增益，然后通过卡尔曼增益按比例缩放残差来更新卡尔曼状态。方程式如下：

$$
\begin{aligned}
\boldsymbol\chi_h &= h(\boldsymbol\chi, u)\\
\mathbf{z}_{mean} &= \frac{1}{N}\sum_1^N \boldsymbol\chi_h \\ \\
\mathbf{P}_{zz} &= \frac{1}{N-1}\sum_1^N [\boldsymbol\chi_h - \mathbf{z}_{mean}][\boldsymbol\chi_h - \mathbf{z}_{mean}]^\mathsf{T} + \mathbf{R} \\
\mathbf{P}_{xz} &= \frac{1}{N-1}\sum_1^N [\boldsymbol\chi - \mathbf{x}^-][\boldsymbol\chi_h - \mathbf{z}_{mean}]^\mathsf{T} \\
\\
\mathbf{K} &= \mathbf{P}_{xz} \mathbf{P}_{zz}^{-1}\\ 
\boldsymbol\chi & = \boldsymbol\chi + \mathbf{K}[\mathbf{z} -\boldsymbol\chi_h + \mathbf{v}_R] \\ \\
\mathbf{x} &= \frac{1}{N} \sum_1^N \boldsymbol\chi \\
\mathbf{P} &= \mathbf{P} - \mathbf{KP}_{zz}\mathbf{K}^\mathsf{T}
\end{aligned}
$$

这与线性KF和UKF非常相似。让我们逐行解释。

第一行，

$$\boldsymbol\chi_h = h(\boldsymbol\chi, u),$$

只需将Sigma点通过测量函数$h$传递即可。我们将结果点命名为$\chi_h$，以便将它们与Sigma点区分开来。在Python中，我们可以这样写：

In [ ]:
sigmas_h = h(sigmas, u)

下一行计算测量Sigma点的平均值。

$$\mathbf{z}_{mean} = \frac{1}{N}\sum_1^N \boldsymbol\chi_h$$

在Python中，我们写作

In [ ]:
z_mean = np.mean(sigmas_h, axis=0)

现在，我们已经得到了测量Sigma点的平均值，我们可以计算每个测量Sigma点的协方差，以及测量Sigma点与Sigma点之间的*交叉协方差*。这由以下两个方程表示：

$$
\begin{aligned}
\mathbf{P}_{zz} &= \frac{1}{N-1}\sum_1^N [\boldsymbol\chi_h - \mathbf{z}_{mean}][\boldsymbol\chi_h - \mathbf{z}_{mean}]^\mathsf{T} + \mathbf{R} \\
\mathbf{P}_{xz} &= \frac{1}{N-1}\sum_1^N [\boldsymbol\chi - \mathbf{x}^-][\boldsymbol\chi_h - \mathbf{z}_{mean}]^\mathsf{T}
\end{aligned}$$

我们可以在Python中用以下方式表示：

In [ ]:
P_zz = 0
for sigma in sigmas_h:
    s = sigma - z_mean
    P_zz += outer(s, s)
P_zz = P_zz / (N-1) + R

P_xz = 0
for i in range(N):
    P_xz += outer(self.sigmas[i] - self.x, sigmas_h[i] - z_mean)
P_xz /= N-1

计算卡尔曼增益很简单，$\mathbf{K} = \mathbf{P}_{xz} \mathbf{P}_{zz}^{-1}$。

在Python中，这个计算如下：

In [ ]:
K = P_xz @ inv(P_zz)

接着，我们用以下公式更新sigma点：

$$\boldsymbol\chi  = \boldsymbol\chi + \mathbf{K}[\mathbf{z} -\boldsymbol\chi_h + \mathbf{v}_R]$$ 

这里的$\mathbf{v}_R$是我们添加到sigma点的扰动。在Python中，我们可以这样实现：

In [ ]:
v_r = multivariate_normal([0]*dim_z, R, N)
for i in range(N):
    sigmas[i] += K @ (z + v_r[i] - sigmas_h[i])

我们的最后一步是重新计算滤波器的均值和协方差。

In [ ]:
    x = np.mean(sigmas, axis=0)
    P = self.P - K @ P_zz @ K.T

## 实现和示例

我已经在`FilterPy`库中实现了一个EnKF。它在很多方面都是一个玩具。使用大量sigma点进行滤波会导致非常慢的性能。此外，文献中有许多算法的细微变化。我写这篇文章主要是因为我对这种滤波器很感兴趣。我没有将其用于现实世界的问题，并且我无法对其用于适用于其适用的大问题的过滤器提供任何建议。因此，我将把我的评论限制在实现一个非常简单的滤波器上。我将使用它来跟踪一个一维对象，并将输出与线性卡尔曼滤波器进行比较。这是我们在本书中已经设计了很多次的滤波器，因此我将以很少的评论来设计它。我们的状态向量将是

$$\mathbf{x} = \begin{bmatrix}x\\ \dot{x}\end{bmatrix}$$

状态转移函数为

$$\mathbf{F} = \begin{bmatrix}1&1\\0&1\end{bmatrix}$$

测量函数为

$$\mathbf{H} = \begin{bmatrix}1&0\end{bmatrix}$$

EnKF是为非线性问题设计的，因此您需要提供Python函数来实现状态转移和测量函数，而不是使用矩阵。 对于这个问题，它们可以编写为：

In [ ]:
def hx(x):
    return np.array([x[0]])

def fx(x, dt):
    return F @ x

最后一件事：与线性Kalman滤波器代码使用的二维列矩阵不同，EnKF代码和UKF代码都使用单个维度$\mathbf{x}$。 

废话不多说，这是代码。

In [ ]:
import kf_book.book_plots as bp
from numpy.random import randn
from filterpy.kalman import EnsembleKalmanFilter as EnKF
from filterpy.kalman import KalmanFilter
from filterpy.common import Q_discrete_white_noise

np.random.seed(1234)

def hx(x):
    return np.array([x[0]])

def fx(x, dt):
    return F @ x
    
F = np.array([[1., 1.],[0., 1.]])

x = np.array([0., 1.])
P = np.eye(2) * 100.
enf = EnKF(x=x, P=P, dim_z=1, dt=1., N=20, hx=hx, fx=fx)

std_noise = 10.
enf.R *= std_noise**2
enf.Q = Q_discrete_white_noise(2, 1., .001)

kf = 卡尔曼滤波器(dim_x=2, dim_z=1)
kf.x = np.array([x]).T
kf.F = F.copy()
kf.P = P.copy()
kf.R = enf.R.copy()
kf.Q = enf.Q.copy()
kf.H = np.array([[1., 0.]])

measurements = []
results = []
ps = []
kf_results = []

zs = []
for t in range (0,100):
    # 创建带有白噪声的测量值 t 
    z = t + randn()*std_noise
    zs.append(z)

In [ ]:
enf.predict()
enf.update(np.asarray([z]))

kf.predict()
kf.update(np.asarray([[z]]))

# 保存数据
results.append (enf.x[0])
kf_results.append (kf.x[0,0])
measurements.append(z)
ps.append(3*(enf.P[0,0]**.5))

results = np.asarray(results)
ps = np.asarray(ps)

In [ ]:
plt.plot(results, label='EnKF')
plt.plot(kf_results, label='KF', c='b', lw=2)
bp.plot_measurements(measurements)
plt.plot(results - ps, c='k',linestyle=':', lw=1, label='1$\sigma$')
plt.plot(results + ps, c='k', linestyle=':', lw=1)
plt.fill_between(range(100), results - ps, results + ps, facecolor='y', alpha=.3)
plt.legend(loc='best');

可以有一点难以看出，但KF和EnKF开始略有不同，但很快就会收敛到产生几乎相同的值。EnKF是一个次优滤波器，因此它不会像KF产生最优解。但是，我故意选择了$N$非常小（20），以确保EnKF输出非常次优。如果我选择一个更合理的数字，比如2000，你将无法在此图上看到两个滤波器输出之间的差异。

## 未解决问题
```

所有这些都应该被视为*我的*问题，而不是文献中的悬而未决的问题。然而，我直接从该领域的知名来源复制方程式，它们没有解决差异。

首先，在Brown [2]中，我们将所有总和乘以$\frac{1}{N}$，如下所示

$$ \hat{x} = \frac{1}{N}\sum_{i=1}^N\chi_k^{(i)}$$

在Crassidis [3]中，相同的方程式为（我将使用与Brown相同的符号，尽管Crassidis的符号不同）

$$ \hat{x} = \frac{1}{N-1}\sum_{i=1}^N\chi_k^{(i)}$$

对于计算协方差的总和，两个来源中的情况都是如此。在谈论滤波器协方差的情况下，Crassidis指出使用$N-1$以确保无偏估计。考虑到以下均值和标准差的标准方程（Crassidis的第2页），这对于协方差是有意义的。

$$
\begin{aligned}
\mu &= \frac{1}{N}\sum_{i=1}^N[\tilde{z}(t_i) - \hat{z}(t_i)] \\
 \sigma^2 &= \frac{1}{N-1}\sum_{i=1}^N\{[\tilde{z}(t_i) - \hat{z}(t_i)] - \mu\}^2
\end{aligned}
$$

然而，我看不出使用$N-1$来计算均值的理由或依据。如果我在均值滤波器中使用$N-1$，滤波器就不会收敛，并且状态基本上会跟随测量值而没有任何滤波。然而，我确实发现在Crassidis中使用$N-1$来计算协方差是有理由的，与Brown不同。同样，我通过经验支持我的决定-在滤波器的实现中，$N-1$是有效的，而$N$不是。

我的第二个问题与$\mathbf{R}$矩阵的使用有关。在Brown中，$\mathbf{R}$被添加到$\mathbf{P}_{zz}$中，而在Crassidis和其他来源中没有。我在网上阅读了其他实现者的笔记，发现添加R有助于滤波器，这对我来说显然是合理和必要的，所以这就是我所做的。

我的第三个问题涉及协方差 $\mathbf{P}$ 的计算。Crassidis 和 Brown 中有不同的方程式。因为 Brown 的实现似乎给了我期望的行为（随着时间的推移，$\mathbf{P}$ 的收敛），并且它与线性 KF 的形式非常相似，所以我选择了 Brown 的实现。相反，我发现 Crassidis 中的方程式似乎不太收敛。

我的第四个问题涉及状态估计更新。在 Brown 中，我们有 

$$\boldsymbol\chi  = \boldsymbol\chi + \mathbf{K}[\mathbf{z} -\mathbf{z}_{mean} + \mathbf{v}_R]$$ 

而在 Crassidis 中，我们有

$$\boldsymbol\chi  = \boldsymbol\chi + \mathbf{K}[\mathbf{z} -\boldsymbol\chi_h + \mathbf{v}_R]$$ 

对我来说，Crassidis 的方程式似乎更合理，对于线性问题，它产生的滤波器的表现类似于线性 KF，因此这是我选择的公式。

我不舒服说这两本书哪个是错的；有可能我错过了一些使每组方程式工作的要点。我可以说，当我按照书上所写的实施时，我没有得到一个可以工作的滤波器。我将“工作”定义为在解决线性问题时基本上与线性卡尔曼滤波器表现相同。在阅读网络上的实现说明和思考各种问题之间，我选择了本章中的实现，它似乎确实可以正确地工作。我还没有探索大量的原始文献，这些文献可能会明确解释这些差异。即使我找到了一个可以调和各种差异的解释，我还是想以某种形式将它留在这里，因为如果我被这些书搞糊涂了，那么其他人可能也会。

## 参考资料

- [1] Mackenzie, Dana. *Ensemble Kalman Filters Bring Weather Models Up to Date* Siam News,  Volume 36, Number 8, October 2003. http://www.siam.org/pdf/news/362.pdf

- [2] Brown, Robert Grover和Patrick Y.C. Hwang. *随机信号与应用卡尔曼滤波的介绍，带有MATLAB®练习和解答。* Wiley，2012。
- [3] Crassidis，John L.和John L. Junkins。 *动态系统的最优估计*。 CRC出版社，2011年。